# 01 - Data preparation (M1)
**Owner of the stage:** data and classification workstream.
Downloads Oxford-IIIT Pet, removes corrupt and duplicate files, groups near-duplicates with perceptual
hashing, and splits 70/15/15 by breed so that no duplicate group crosses splits. Full-body boxes and
masks are derived from the trimaps. The split is saved to `runs/data/metadata.csv` and reused by every
later notebook.

In [ ]:
# Setup — mount Drive before resolving the repository.
import os, sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")

def _resolve_spv_repo(search_root: Path, explicit: str | None = None) -> Path:
    def valid(path):
        return (path / "src" / "spv").is_dir() and (path / "configs" / "config.yaml").is_file()
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if not valid(candidate):
            raise FileNotFoundError(f"SPV_REPO is not a project root: {candidate}")
        return candidate
    candidates = set()
    for pattern in ("smart-pet-vision", "*smart-pet-vision", "*/smart-pet-vision", "*/*/smart-pet-vision"):
        candidates.update(path.resolve() for path in search_root.glob(pattern) if valid(path))
    if valid(search_root):
        candidates.add(search_root.resolve())
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Expected one smart-pet-vision repository under {search_root}; found {len(candidates)}. "
            "Add the shared folder shortcut to My Drive, or set SPV_REPO to its exact path."
        )
    return candidates.pop()

if IN_COLAB:
    REPO = _resolve_spv_repo(Path("/content/drive/MyDrive"), os.environ.get("SPV_REPO"))
else:
    explicit = os.environ.get("SPV_REPO")
    local_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                       if (path / "src" / "spv").is_dir()), Path.cwd())
    REPO = _resolve_spv_repo(local_root, explicit)
os.chdir(REPO)
if IN_COLAB:
    !pip -q install -r requirements.txt
sys.path.insert(0, os.path.abspath("src"))

from spv.common import load_config, seed_everything, save_json, load_json
cfg = load_config()
if IN_COLAB:  # images on fast local disk (re-downloaded each session); results stay on Drive
    cfg["paths"]["raw"] = Path("/content/data/raw")
    cfg["paths"]["processed"] = Path("/content/data/processed")
seed_everything(cfg["seed"])
RUNS = Path(cfg["paths"]["runs"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
from spv import data
df = data.ensure_data(cfg)          # first run: download + clean + split (about 10 min)
IMG = data.dataset_dir(cfg) / "images"
PROC = Path(cfg["paths"]["processed"])
print(len(df), "images after cleaning")

## Cleaning and de-duplication report

In [ ]:
import pandas as pd
dropped = pd.read_csv(RUNS / "data" / "dropped.csv")
near = pd.read_csv(RUNS / "data" / "near_duplicates.csv")
print("Dropped by reason:"); print(dropped.drop_reason.value_counts().to_string())
print("\nNear-duplicate pairs (pHash Hamming <=", cfg["data"]["phash_threshold"], "):", len(near))
print("Duplicate groups with more than one image:", int((df.groupby("group").size() > 1).sum()))
near.sort_values("hamming").head(10)

## Split and leakage audit

In [ ]:
print(data.split_summary(df).to_string())
audit = data.leakage_audit(df); print(audit)
assert audit["passed"], "A duplicate group crosses splits - do not continue"
breed_split = pd.crosstab(df.breed, df.split)
breed_split.head(37)

## Image statistics

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
df.species.value_counts().plot.bar(ax=ax[0], title="Images per species")
df.breed.value_counts().plot.bar(ax=ax[1], title="Images per breed", fontsize=6)
(df.box_fraction if "box_fraction" in df else df.mask_fraction).plot.hist(bins=40, ax=ax[2], title="Body box area / image area")
plt.tight_layout(); fig.savefig(RUNS / "data" / "dataset_stats.png", dpi=150)
df[["width", "height", "mask_fraction"]].describe().round(3)

## Manual review sheets
Two random samples were reviewed on the contact sheets below and recorded in `runs/data/review/<kind>/review.csv`
(columns `ok` = 1 correct / 0 flagged, `note`, `reviewer`, `owner_confirmed`):
* **labels** - 200 images: is the species label right? (bounds the label-error rate)
* **boxes** - 300 images: does the red box enclose the whole animal? (validates the trimap-derived boxes)

The review was done visually on 19 Oct 2026 and confirmed by the team. The cell after the sheets summarises it.

In [ ]:
import shutil
for kind, n in (("labels", cfg["data"]["audit_sample"]), ("boxes", cfg["data"]["box_review_sample"])):
    folder = data.review_sheets(data.dataset_dir(cfg), df, PROC, n, kind, cfg["seed"])
    dst = RUNS / "data" / "review" / kind
    if not (dst / "review.csv").exists():
        shutil.copytree(folder, dst, dirs_exist_ok=True)
    print(kind, "->", dst)
from IPython.display import Image as Show
Show(str(RUNS / "data" / "review" / "boxes" / "sheet_00.jpg"), width=900)

In [ ]:
# Summary of the completed reviews (ok: 1 = correct, 0 = flagged; y/n is also accepted)
import math
def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0.0, c - h), min(1.0, c + h)
CODES = {"1": 1, "y": 1, "yes": 1, "0": 0, "n": 0, "no": 0}
for kind in ("labels", "boxes"):
    r = pd.read_csv(RUNS / "data" / "review" / kind / "review.csv", keep_default_na=False)
    ok = r["ok"].astype(str).str.strip().str.lower().map(CODES)
    done = r[ok.notna()]
    if len(done) == 0:
        print(f"{kind}: not reviewed yet")
        continue
    n, k = len(done), int((ok.dropna() == 0).sum())
    lo, hi = wilson(k, n)
    line = f"{kind}: {n} reviewed, {k} flagged ({k / n:.1%}, 95% Wilson CI {lo:.1%}-{hi:.1%}"
    if k == 0:
        line += f"; one-sided 95% upper bound {1 - 0.05 ** (1 / n):.1%}"
    confirmed = int((r.get("owner_confirmed", "").astype(str).str.strip() != "").sum()) if "owner_confirmed" in r else 0
    print(line + f"), owner-confirmed rows {confirmed}/{len(r)}")
    for row in done[done["note"].astype(str).str.strip() != ""].itertuples():
        print(f"   #{row.idx} {row.image} ok={row.ok}: {row.note}")

## Values for the dataset card (`docs/dataset_card.md`)

In [ ]:
card = {"images_after_cleaning": len(df), "cats": int((df.species=="cat").sum()),
        "dogs": int((df.species=="dog").sum()), "breeds": df.breed.nunique(),
        "dropped": dropped.drop_reason.value_counts().to_dict(), "near_duplicate_pairs": len(near),
        "split_counts": df.split.value_counts().to_dict(), "leakage_audit": audit,
        "median_size": [int(df.width.median()), int(df.height.median())]}
save_json(card, RUNS / "data" / "dataset_card_values.json"); card